## Importiamo il dataset

In [1]:
from datasets import load_dataset

# Carica il dataset 
dataset = load_dataset("json", data_files="data/c4-en-10k.json")
print(dataset)
#Salviamo il testo per allenare il tokenizer
with open("data/corpus_c4.txt", "w", encoding="utf-8") as f:
    for text in dataset["train"]["text"]:
        f.write(text + "\n")

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 10000
    })
})


## Creiamo il tokenizer

In [2]:
from tokenizers import SentencePieceBPETokenizer

# Inizializza il tokenizer
tokenizer = SentencePieceBPETokenizer()

# Definiamo i token speciali 
special_tokens = ["<pad>", "</s>", "<unk>"]
sentinel_tokens = [f"<extra_id_{i}>" for i in range(100)]
all_special_tokens = special_tokens + sentinel_tokens

# Addestriamo il tokenizer
tokenizer.train(
    files=["data/corpus_c4.txt"],
    vocab_size=32128, # Dimensione standard di T5
    min_frequency=2,
    show_progress=True,
    special_tokens=all_special_tokens
)

# Salviamo il tokenizer
tokenizer.save("./tokenizers/t5-c4-tokenizer.json")

## Carichiamo il vocabolario in un oggetto T5TokenizerFast di huggingface

In [3]:
from transformers import T5TokenizerFast

tokenizer = T5TokenizerFast(
    tokenizer_file="./tokenizers/t5-c4-tokenizer.json",
    bos_token="<s>", 
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    extra_ids=100  # Fondamentale per T5 e i suoi <extra_id_0>...
)

# Test rapido
test_text = "This is a test for T5 pre training"
encoded = tokenizer.encode(test_text)
print(f"Token IDs: {encoded}")
print(f"Decodifica: {tokenizer.decode(encoded)}")

Token IDs: [1419, 1163, 1104, 2226, 1165, 20879, 1493, 2802, 1]
Decodifica: This is a test for T5 pre training</s>


## Definiamo l'architettura del modello

In [4]:
from transformers import T5Config, T5ForConditionalGeneration

In [5]:
config = T5Config(
    vocab_size=len(tokenizer),           # Deve corrispondere al tuo tokenizer SentencePiece
    d_model=512,                # Dimensione delle rappresentazioni nascoste
    d_kv=64,                    # Dimensione delle chiavi/valori nell'attenzione
    d_ff=2048,                  # Dimensione dello strato Feed-Forward
    num_layers=6,               # Numero di blocchi nell'Encoder
    num_decoder_layers=6,       # Numero di blocchi nel Decoder
    num_heads=8,                # Numero di teste di Multi-Head Attention
    relative_attention_num_buckets=32,
    dropout_rate=0.1,
    initializer_factor=1.0,     # Importante per l'inizializzazione dei pesi da zero
    feed_forward_proj="relu",   # T5 originale usa ReLU o GeGLU gated-relu
    is_encoder_decoder=True,
    decoder_start_token_id=tokenizer.pad_token_id
)

# Inizializzazione del modello con pesi casuali
model = T5ForConditionalGeneration(config)

## Definiamo la logica di masking delle parole con i sentinel

In [6]:
import numpy as np

def tokenize_and_mask(text, 
                      noise=0.15, 
                      randomizer=np.random.uniform, 
                      tokenizer=None):
    """Tokenizes and masks a given input for T5 pre-training."""
    
    # Numero del sentinel corrente (partiamo da 0)
    cur_sentinel_num = 0
    
    # Liste per input e target
    inps, targs = [], []

    # Vocab_size (necessario per calcolare gli ID dei sentinel)
    vocab_size = int(tokenizer.vocab_size)
    
    # EOS token id 
    eos = tokenizer.convert_tokens_to_ids("</s>")
    
    
    # prev_no_mask è True se il token precedente NON era mascherato
    prev_no_mask = True
    
    # Tokenizzazione (assumendo l'uso di Hugging Face tokenizer o simili)
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    for token in tokens:
        # Genera un valore casuale tra 0 e 1
        rnd_val = randomizer() 
        
        # Se il rumore è maggiore del valore random, mascheriamo (coin flip)
        if noise > rnd_val:
            
            # Se il precedente NON era mascherato, iniziamo un nuovo span
            if prev_no_mask:
                
                # Calcoliamo l'ID del sentinel corrente (es: 32127, 32126...)
                # I sentinel in T5 sono <extra_id_0>, <extra_id_1>...
                # mappati solitamente in fondo al vocabolario.
                cur_sentinel_num += 1
                end_id = tokenizer.convert_tokens_to_ids(f"<extra_id_{cur_sentinel_num}>")
                
                # Aggiungiamo il sentinel sia all'input che al target
                # L'input riceve il sentinel come "segnaposto"
                inps.append(end_id)
                # Il target riceve il sentinel per marcare l'inizio dello span rimosso
                targs.append(end_id)
                
            # Aggiungiamo sempre il token originale ai target (lo span da ricostruire)
            targs.append(token)
            
            # Il token corrente è mascherato, quindi prev_no_mask diventa False
            prev_no_mask = False

        else:
            # Se NON mascheriamo, il token va dritto negli input
            inps.append(token)
            
            # Il token corrente non è mascherato, quindi prev_no_mask diventa True
            prev_no_mask = True
    
    # Ogni sequenza target di T5 deve finire con EOS
    targs.append(eos)
    
    ### END CODE HERE ###
    
    return inps, targs

In [7]:
input_str = 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers.'

inps, targs = tokenize_and_mask(input_str, tokenizer=tokenizer)
print(f"tokenized inputs - shape={len(inps)}:\n\n{tokenizer.decode(inps, skip_special_tokens=False)}\n\ntargets - shape={len(targs)}:\n\n{tokenizer.decode(targs, skip_special_tokens=False)}")

tokenized inputs - shape=53:

Beginners BBQ Class Taking Place in Miss<extra_id_1>a!
Do you want to get<extra_id_2> at making delicious BBQ? You will<extra_id_3> the opportunity, put this<extra_id_4> calendar now.<extra_id_5> September 22nd join World Class BBQ Champion, Tony<extra_id_6> from Lonestar Smoke<extra_id_7>ers.

targets - shape=17:

<extra_id_1>oul<extra_id_2> better<extra_id_3> have<extra_id_4> on your<extra_id_5> Thursday,<extra_id_6> Balay<extra_id_7> Rang</s>


## Applichiamo la funzione al dataset

In [8]:
import numpy as np

def preprocess_function(examples):
    # Liste per i risultati del batch
    batch_inputs = []
    batch_targets = []
    
    for text in examples["text"]:
        # Usiamo la funzione che abbiamo completato prima
        inps, targs = tokenize_and_mask(
            text, 
            noise=0.15, 
            tokenizer=tokenizer
        )
        batch_inputs.append(inps)
        batch_targets.append(targs)
    
    return {
        "input_ids": batch_inputs,
        "labels": batch_targets
    }

# Applichiamo la funzione al dataset (rimuovendo la colonna 'text' originale)
tokenized_c4 = dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## Definiamo il data loader

In [10]:
import torch
def pad_collate_fn(batch):
    # 1. TRONCAMENTO: Fondamentale per non andare in OOM
    # T5 pre-training standard usa 512 per gli input e 128 per i target
    max_input_len = 512 
    max_target_len = 128
    
    input_ids = [torch.tensor(item["input_ids"][:max_input_len]) for item in batch]
    labels = [torch.tensor(item["labels"][:max_target_len]) for item in batch]
    
    # 2. PADDING: Rende tutti i campioni della stessa lunghezza del più lungo NEL BATCH
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    
    labels_padded = torch.nn.utils.rnn.pad_sequence(
        labels, batch_first=True, padding_value=-100
    )
    
    # 3. ATTENTION MASK: Dice al modello quali token ignorare
    return {
        "input_ids": input_ids_padded,
        "labels": labels_padded,
        "attention_mask": (input_ids_padded != tokenizer.pad_token_id).long()
    }

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_c4["train"], 
    batch_size=4, 
    shuffle=True, 
    collate_fn=pad_collate_fn,
)

# Test rapido: prendiamo un batch
sample_batch = next(iter(train_dataloader))
print(f"Shape Input IDs: {sample_batch['input_ids'].shape}")
print(f"Shape Labels: {sample_batch['labels'].shape}")

Shape Input IDs: torch.Size([4, 512])
Shape Labels: torch.Size([4, 128])


## Definisco una funzione per testare i risultati

In [26]:
def monitor_progress(text_masked, tokenizer, model, device):
    model.eval()
    inputs = tokenizer(text_masked, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"], 
            max_length=50,
            do_sample=True, 
            top_k=50
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"\n[Monitor] Input: {text_masked}")
    print(f"[Monitor] Generazione: {decoded}\n")
    model.train()

# Esempio di test
test_sentence = "The capital of Italy is <extra_id_1> and it is a <extra_id_1> city."

## Definiamo iperparametri ottimizzatore e loss

In [30]:
from transformers import get_linear_schedule_with_warmup
import torch.optim as optim

# Parametri
learning_rate = 5e-4
epochs = 3
grad_acc_steps = 4 

optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

# Scheduler: aumenta il LR gradualmente all'inizio per evitare divergenza
total_steps = len(train_dataloader) * epochs // grad_acc_steps
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=1000, num_training_steps=total_steps
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32129, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32129, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [28]:
monitor_progress(test_sentence, tokenizer, model, 'cuda')


[Monitor] Input: The capital of Italy is <extra_id_1> and it is a <extra_id_1> city.
[Monitor] Generazione: <pad><extra_id_1> that<extra_id_2> with</s>



## Inizio l'addestramento

In [31]:
import time

print("Inizio addestramento...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for step, batch in enumerate(train_dataloader):
        # Sposta i dati sulla GPU
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        # Forward pass
        # Hugging Face T5 calcola internamente la CrossEntropy se passi labels
        outputs = model(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels
        )
        
        loss = outputs.loss / grad_acc_steps
        loss.backward()
        
        total_loss += loss.item()
        
        # Aggiornamento pesi (Gradient Accumulation)
        if (step + 1) % grad_acc_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Evita gradienti esplosivi
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        # Ogni 500 step, vediamo come genera
        if step % 100 == 0:
            avg_loss = total_loss / (step + 1) * grad_acc_steps
            print(f"Epoch {epoch} | Step {step} | Loss: {avg_loss:.4f}")
            monitor_progress(test_sentence, tokenizer, model, device)

    print(f"Epoch {epoch} completata in {time.time() - start_time:.2f}s")

Inizio addestramento...
Epoch 0 | Step 0 | Loss: 4.9286

[Monitor] Input: The capital of Italy is <extra_id_1> and it is a <extra_id_1> city.
[Monitor] Generazione: <pad><extra_id_1> for<extra_id_2> This<extra_id_3>.<extra_id_4> the<extra_id_5>.<extra_id_6> to</s>

Epoch 0 | Step 100 | Loss: 4.8213

[Monitor] Input: The capital of Italy is <extra_id_1> and it is a <extra_id_1> city.
[Monitor] Generazione: <pad><extra_id_1> an<extra_id_2> are<extra_id_3> or<extra_id_4> to<extra_id_5></s>

Epoch 0 | Step 200 | Loss: 4.8092

[Monitor] Input: The capital of Italy is <extra_id_1> and it is a <extra_id_1> city.
[Monitor] Generazione: <pad><extra_id_1> with<extra_id_2> of<extra_id_3> to<extra_id_4> and<extra_id_5>
<extra_id_6> with</s>

Epoch 0 | Step 300 | Loss: 4.7851

[Monitor] Input: The capital of Italy is <extra_id_1> and it is a <extra_id_1> city.
[Monitor] Generazione: <pad><extra_id_1> my<extra_id_2> to<extra_id_3>.</s>

Epoch 0 | Step 400 | Loss: 4.7734

[Monitor] Input: The capital

KeyboardInterrupt: 